In [ ]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")
df = pd.merge(ratings, movies, on="movieId")
df = df[['userId', 'movieId', 'rating', 'title', 'genres']]

print(df.head().to_string(index=False))

In [ ]:
user_movie_matrix = df.pivot_table(
    index='userId',
    columns='title',
    values='rating'
).fillna(0)

user_movie_matrix.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

In [ ]:
def get_top_movies(n=5):
    avg_rating = df.groupby('title')['rating'].mean()
    rating_count = df.groupby('title')['rating'].count()

    top = pd.DataFrame({
        'avg_rating': avg_rating,
        'rating_count': rating_count
    })

    top = top[top['rating_count'] > 50]   # keep popular ones
    top = top.sort_values(by='avg_rating', ascending=False)

    return top.head(n).index.tolist()

In [ ]:
def recommend_by_genre(movie_name):
    genre = df[df['title'] == movie_name]['genres'].values
    
    if len(genre) == 0:
        return []
    
    genre_set = set(genre[0].split('|'))

    def match(x):
        return len(genre_set.intersection(set(x.split('|'))))

    df['score'] = df['genres'].apply(match)

    recs = df[df['score'] >= 2]
    recs = recs[recs['title'] != movie_name]

    return recs.sort_values(by='score', ascending=False)['title'].unique()[:5]

In [ ]:
def recommend_users(user_id):
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)[1:6]

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    recs = {}

    for sim_user in similar_users.index:
        sim_movies = user_movie_matrix.loc[sim_user]

        for movie in sim_movies.index:
            if movie not in watched and sim_movies[movie] > 0:
                recs[movie] = recs.get(movie, 0) + sim_movies[movie]

    sorted_movies = sorted(recs.items(), key=lambda x: x[1], reverse=True)

    return [m[0] for m in sorted_movies[:5]]

In [ ]:
def get_movies_by_genre(genre):
    g = df[df['genres'].str.contains(genre)]
    top = g.groupby('title')['rating'].mean().sort_values(ascending=False)
    return top.head(5).index.tolist()

In [ ]:
search_history = []
def search_movies(name):
    global search_history

    result = df[df['title'].str.contains(name, case=False)]
from collections import Counter

def get_top_searched(n=5):
    return [i[0] for i in Counter(search_history).most_common(n)]
query = input("🔍 Search Movie: ")

if query == "":
    print("🔥 Top Movies:")
    print(get_top_movies())

else:
    results = search_movies(query)

    print("\n🔍 Search Results:")
    for m in results:
        print("🎬", m)